# Wasserstein barycenter grids in one and two dimensions

This notebook generates `fig:barycenters-four-shapes`.  It displays two complementary barycenter grids for four corner measures.  In one dimension, the quadratic Wasserstein barycenter is obtained by bilinear averaging of quantile functions,
$$
    Q_{u,v}=\sum_{i,j\in\{0,1\}} \lambda_{ij}(u,v)Q_{ij}.
$$
In two dimensions, a shared reference cloud is mapped to four simple shapes, and the same bilinear weights average the four displacement maps.  This gives a lightweight, correspondence-based visual proxy for Wasserstein shape barycenters.

In [1]:
from pathlib import Path
import os
import sys
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")
for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        ROOT = candidate.parent if candidate.name == "notebooks-figures" else candidate
        break
else:
    raise RuntimeError("Could not locate figure_style.py")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Ellipse
from PIL import Image
import ot
from scipy.linalg import expm, solve
from scipy.spatial.distance import cdist
from scipy.ndimage import gaussian_filter
from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY, DIRAC_MARKER_SIZE,
    setup_matplotlib, figure_dir, save_pdf, remove_axes, box_axes,
    interp_color, draw_point_clouds, draw_transport_segments, padded_limits,
)
setup_matplotlib()
rng = np.random.default_rng(2027)

In [2]:
NAME = "barycenters-four-shapes"
out = figure_dir(NAME)

corner_colors = np.array([
    [0.839, 0.188, 0.153],
    [0.992, 0.682, 0.380],
    [0.482, 0.196, 0.580],
    [0.129, 0.400, 0.675],
])
def weights(u, v):
    return np.array([(1-u)*(1-v), u*(1-v), (1-u)*v, u*v])
def bilinear_color(u, v):
    return tuple(np.clip(weights(u,v) @ corner_colors, 0, 1))

# 1D quantile barycenter grid.
s = np.linspace(0.01, 0.99, 160)
Q = np.vstack([
    -1.25 + 0.36*np.tan(np.pi*(s-0.5)) / 2.7,
    -0.20 + 0.72*np.tan(np.pi*(s-0.5)) / 2.7,
     0.25 + 0.30*np.tan(np.pi*(s-0.5)) / 2.7 + 0.22*np.sin(2*np.pi*s),
     1.10 + 0.52*np.tan(np.pi*(s-0.5)) / 2.7,
])
Q = np.clip(Q, -2.1, 2.1)
uv = np.linspace(0,1,5)
fig, ax = plt.subplots(figsize=(3.0, 3.0))
for row, v in enumerate(uv[::-1]):
    for col, u in enumerate(uv):
        q = weights(u,v) @ Q
        x0, y0 = col, row
        hist, edges = np.histogram(q, bins=42, range=(-2.15,2.15), density=True)
        hist = hist / max(hist.max(), 1e-12)
        xs = (edges[:-1] + edges[1:]) / 2
        xs = x0 + 0.78*(xs - xs.min())/(xs.max()-xs.min()) + 0.10
        ys = y0 + 0.12 + 0.70*hist
        ax.fill_between(xs, y0+0.12, ys, color=bilinear_color(u,v), alpha=0.28, linewidth=0)
        ax.plot(xs, ys, color=bilinear_color(u,v), lw=0.80)
ax.set_xlim(-0.05, 5.0); ax.set_ylim(-0.05,5.0); ax.set_aspect('equal'); remove_axes(ax)
save_pdf(fig, out/'quantile-grid.pdf', pad_inches=0.045); plt.close(fig)

# 2D shape barycenter grid.
N = 180
r = np.sqrt(rng.random(N)); th = 2*np.pi*rng.random(N)
p = np.c_[r*np.cos(th), r*np.sin(th)]
def polar(q): return np.linalg.norm(q,axis=1), np.arctan2(q[:,1],q[:,0])
def rot(a):
    c,s=np.cos(a),np.sin(a); return np.array([[c,-s],[s,c]])
def disk(q):
    rad,ang=polar(q); R=0.82+0.06*np.cos(5*ang)
    return np.c_[R*rad*np.cos(ang), R*rad*np.sin(ang)]
def ellipse(q):
    z=q@rot(0.26).T; return np.c_[1.35*z[:,0],0.48*z[:,1]]
def heart(q):
    rad,ang=polar(q); R=0.55+0.32*(1+np.sin(ang))
    return np.c_[R*rad*np.sin(ang), 0.88*R*rad*np.cos(ang)-0.18*rad]
def two_lobes(q):
    z=q.copy(); side=np.sign(z[:,0]+1e-8); z[:,0]=0.62*z[:,0]+0.34*side; z[:,1]*=0.72
    return z
corners=[disk(p), ellipse(p), heart(p), two_lobes(p)]
def cloud(u,v): return sum(w*S for w,S in zip(weights(u,v),corners))
def normalize(q):
    q=q-q.mean(0); return q/max(np.linalg.norm(q,axis=1).max(),1e-12)
fig, ax = plt.subplots(figsize=(3.0, 3.0))
for row, v in enumerate(uv[::-1]):
    for col, u in enumerate(uv):
        z = 0.37*normalize(cloud(u,v)) + np.array([col+0.5,row+0.5])
        ax.scatter(z[:,0], z[:,1], s=DIRAC_MARKER_SIZE*0.42, marker='o', color=bilinear_color(u,v), edgecolor='none', linewidth=0, alpha=0.86)
ax.set_xlim(0,5); ax.set_ylim(0,5); ax.set_aspect('equal'); remove_axes(ax)
save_pdf(fig, out/'shape-grid.pdf', pad_inches=0.045); plt.close(fig)